In [ ]:
from datetime import datetime
from getpass import getpass

rdm_url = 'https://develop.rdm.example.com/'

idp_name_1 = None
idp_username_institutional_admin = 'user_test_admin'
idp_password_institutional_admin = 'password_test_admin'
idp_username_2 = "user_test02"
idp_password_2 = "password_test02"
idp_username_4 = 'user_test04'
idp_password_4 = 'password_test04'

display_username_institutional_admin = 'name_admin_user'
display_username_1 = 'name_user_test01'
display_username_2 = 'name_user_test02'
display_username_3 = 'name_user_test03'

group_a = 'GroupA'
group_b = 'GroupB'
group_c = 'GroupC'
group_d = 'GroupD'

group_search = 'Group'
target_storage_name = 'NII Storage'
target_storage_id = 'osfstorage'
rdm_project_name = 'TEST-グループ管理連携機能検証-{}'.format(datetime.now().strftime('%Y%m%d'))
delete_project = True
default_result_path = None
close_on_fail = False
transition_timeout = 60000
group_note_text = '※本機能はムーンショット目標2'

In [ ]:
if idp_username_institutional_admin is None:
    idp_username_institutional_admin = input(prompt=f'Username for {idp_name_1}')
if idp_password_institutional_admin is None:
    idp_password_institutional_admin = getpass(prompt=f'Password for {idp_username_institutional_admin}@{idp_name_1}')
(len(idp_username_institutional_admin), len(idp_password_institutional_admin))

In [ ]:
if idp_username_2 is None:
    idp_username_2 = input(prompt=f'Username for {idp_name_1}')
if idp_password_2 is None:
    idp_password_2 = getpass(prompt=f'Password for {idp_username_2}@{idp_name_1}')
(len(idp_username_2), len(idp_password_2))

if idp_username_4 is None:
    idp_username_4 = input(prompt=f'Username for {idp_name_1}')
if idp_password_4 is None:
    idp_password_4 = getpass(prompt=f'Password for {idp_username_4}@{idp_name_1}')
(len(idp_username_4), len(idp_password_4))

In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir

# グループ権限によるコンポーネントに対する操作

- サブシステム名: グループ管理連携機能
- ページ/アドオン: グループ権限によるコンポーネントへのアクセス
- 機能分類: グループ権限によるコンポーネントに対する操作
- シナリオ名: コンポーネントに対する権限確認
- 用意するテストデータ: アカウント(機関管理者1,既存ユーザー1,2,3,4,5),再利用可能(条件なし)

In [ ]:
import importlib
import pandas as pd

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *
from scripts import grdm

await init_pw_context(close_on_fail=False, last_path=default_result_path)

## ウェブブラウザの新規プライベートウィンドウで GRDM トップページを表示する

GRDM トップページが表示されること

In [ ]:
async def _step(page):
    await page.goto(rdm_url)

    # 同意する をクリック
    await page.locator('//button[text() = "同意する"]').click()

    # 同意する が表示されなくなったことを確認
    await expect(page.locator('//button[text() = "同意する"]')).to_have_count(0, timeout=500)

await run_pw(_step)

## RDMIdPを利用し、既存ユーザー2としてログインする

- GRDM ダッシュボードが表示されること
- 「プロジェクトに対するグループ機能①」シートのNo.4で追加したプロジェクトが表示されていること
- 該当プロジェクトの「Groups」項目に既存ユーザー2が所属しているグループ(グループA)が表示されていること

In [ ]:
async def _step(page):
    await scripts.grdm.login(page, idp_name_1, idp_username_2, idp_password_2, transition_timeout=transition_timeout)

    project_locator = page.locator('[data-test-dashboard-item]', has=page.locator('[data-test-dashboard-item-title]', has_text=rdm_project_name)).first
    await expect(project_locator).to_be_visible(timeout=transition_timeout)
    await expect(project_locator).to_contain_text(group_a, timeout=transition_timeout)

await run_pw(_step)

## "ダッシュボードから「プロジェクトに対するグループ機能①」シートのNo.4"で作成したプロジェクトをクリックする"
- プロジェクトダッシュボードが表示されること
- プロジェクトダッシュボードの上部メニューに「ファイル」、「Wiki」、「メタデータ」、「エクスポート」、「メンバ」、「グループ」、「アドオン」、「設定」が表示されること(「読込み/書込み」権限)

In [ ]:
async def _step(page):
    await page.locator(f'//*[@data-test-dashboard-item-title and text()="{rdm_project_name}"]').click()

    await expect(page.locator("#projectSubnav").get_by_role("link", name="ファイル")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="Wiki")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="メタデータ")).to_be_visible(timeout=transition_timeout)
    # await expect(page.locator("#projectSubnav").get_by_role("link", name="エクスポート")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="メンバー")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="グループ", exact=True)).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="アドオン")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="設定")).to_be_visible(timeout=transition_timeout)
    await expect(grdm.get_select_expanded_storage_title_locator(page, target_storage_name)).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(1)

await run_pw(_step)

## プロジェクトダッシュボードの画面右部の「コンポーネント」の「コンポーネントを追加」をクリックする

- 「新しいコンポーネントを作成する」ダイアログが表示されること

In [ ]:
async def _step(page):
    # Click the "コンポーネントを追加" (Add Component) button
    add_component_button = page.locator('div[data-toggle="modal"][data-target="#addSubComponent"]')
    await expect(add_component_button).to_be_visible(timeout=transition_timeout)
    await add_component_button.click()
    
    # Verify the "新しいコンポーネントを作成する" (Create New Component) dialog appears
    await asyncio.sleep(1)
    await expect(page.locator('text=新しいコンポーネントを作成する')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## タイトル「TEST-グループ管理連携機能検証-YYYYMMDD(本日の日付)-コンポーネント-ユーザー2」でコンポーネントを作成する
※上記のコンポーネントを作成する際、「次からメンバーとグループを追加する：{プロジェクト名}」にチェックを入れない

- 「新しいコンポーネントが作成されました！」ダイアログが表示されること

In [ ]:
rdm_component_name = 'TEST-グループ管理連携機能検証-{}-コンポーネント-ユーザー2'.format(datetime.now().strftime('%Y%m%d'))

async def _step(page):
    # Input the component name
    project_name_input = page.locator('#addSubComponent input.form-control.project-name[name="projectName"]')
    await expect(project_name_input).to_be_visible(timeout=transition_timeout)
    await project_name_input.type(rdm_component_name)
    await asyncio.sleep(1)

    # Click the save button (作成)
    save_button = page.locator('#addSubComponent button.btn-success:has-text("作成")')
    await expect(save_button).to_be_enabled(timeout=transition_timeout)
    await save_button.click()
    
    # Verify the success dialog appears
    success_dialog = page.locator('text=新しいコンポーネントが作成されました！')
    await expect(success_dialog).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「新しいコンポーネントが作成されました！」ダイアログの「新しいコンポーネントへ移動する」をクリックする

- 作成したコンポーネントのプロジェクトダッシュボードが表示されること

In [ ]:
async def _step(page):
    # Click "新しいコンポーネントへ移動する" button in the success dialog
    go_to_component_link = page.locator('a.btn.btn-success:has-text("新しいコンポーネントへ移動する")')
    await expect(go_to_component_link).to_be_visible(timeout=transition_timeout)
    await go_to_component_link.click()

    # Wait for navigation to the new component's dashboard
    await page.wait_for_load_state('networkidle')
    
    # Verify that the component's project dashboard is displayed
    # Check for the component name in the title
    await expect(page.locator('span#nodeTitleEditable')).to_contain_text(rdm_component_name, timeout=transition_timeout)
await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから「アドオン」をクリックする

- アドオン設定画面が表示されること

In [ ]:
async def _step(page):
    await page.locator("#projectSubnav").get_by_role("link", name="アドオン").click()
    await expect(page.get_by_role("heading", name="アドオンを選択", level=3)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「アドオンを選択」のパネル内「Groups」の行の「有効にする」をクリックする。

・「Groups アドオン規約」のダイアログが表示されること

In [ ]:
async def _step(page):
    enable_button = page.locator('div.addon-container[name="groups"] a', has_text="有効にする")
    await enable_button.scroll_into_view_if_needed()
    await enable_button.click()
    await expect(page.get_by_role("heading", name="Groups アドオン規約", level=3)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「確認」をクリックする

- 「アドオンを構成」のパネル内に「Groups」の行が追加されること
-  プロジェクトダッシュボードの上部メニューに 「グループ」が追加されること

In [ ]:
async def _step(page):
    # Locate and click the "確認" button
    confirm_button = page.locator('button[data-bb-handler="confirm"]', has_text="確認")
    await confirm_button.scroll_into_view_if_needed()
    await confirm_button.click()
    await expect(page.locator('div#groupsScope.scripted')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="グループ", exact=True)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 作成したコンポーネントのプロジェクトダッシュボードの上部メニューから「メンバー」をクリックする

- 「メンバー」画面が表示されること
- 各メンバーが以下の通りとなっていること。
  - 既存ユーザー2 : 管理者

In [ ]:
async def _step(page):
    await page.locator("#projectSubnav").get_by_role("link", name="メンバー").click()
    await expect(page.get_by_role("heading", name="メンバー", level=3)).to_be_visible(timeout=transition_timeout)
    # Verify member row with specific user and permission
    existing_user_row = page.locator('tr.contrib').filter(has_text=display_username_2)
    await expect(existing_user_row).to_be_visible()
    
    # Verify the permission text is "管理者" (Administrator)
    await expect(existing_user_row.locator('.permission-search')).to_have_text('管理者')

await run_pw(_step)

## 作成したコンポーネントのプロジェクトダッシュボードの上部メニューから「グループ」をクリックする

- 「グループ」画面が表示されること
- 各グループに設定されているグループが存在しないこと

In [ ]:
async def _step(page):
    await page.locator("#projectSubnav").get_by_role("link", name="グループ", exact=True).click()
    await expect(page.get_by_role("heading", name=group_note_text)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから「{プロジェクト名}」をクリックする

- プロジェクトダッシュボードが表示されること
- プロジェクトダッシュボードの画面右部の「コンポーネント」の「{コンポーネント名}」の下に、以下のメンバが表示されること
  - 既存ユーザー2 

In [ ]:
async def _step(page):
    # Click project name in top menu to return to project dashboard
    await page.locator('a[data-toggle="tooltip"][title*="TEST-グループ管理連携機能検証"]').click()
    await page.wait_for_load_state('networkidle')
    await asyncio.sleep(1)

    # Wait for components widget to be visible
    components_section = page.locator("span#components, .render-nodes-list, div:has-text('コンポーネント')")
    await expect(components_section.first).to_be_visible(timeout=transition_timeout)

    # Verify component exists by checking for any component item
    component_item = components_section.locator("li.list-group-item-node").first
    await expect(component_item).to_be_visible(timeout=transition_timeout)
    
    # Verify component title contains part of the expected text
    component_title = component_item.locator("h4.list-group-item-heading a").first
    await expect(component_title).to_be_visible(timeout=transition_timeout)
    
    # Verify the title contains expected keywords
    title_text = await component_title.text_content()
    assert "コンポーネント" in title_text, f"Expected 'コンポーネント' in title, but got: {title_text}"
    
    # Verify member is displayed under the component
    member_link = component_item.locator(".project-authors a.overflow")
    await expect(member_link).to_be_visible(timeout=transition_timeout)
    await expect(member_link).to_have_text(display_username_2)
    
    # Alternative: verify by tooltip attribute
    await expect(member_link).to_have_attribute("data-original-title", display_username_2)

await run_pw(_step)

## 以下の手順を実施し、コンポーネントを削除する
- コンポーネントのプロジェクトダッシュボードの上部メニューから「設定」をクリックする
- 「コンポーネントを削除」を選択する
- 「次の文字列を入力して続行します」に記載されている文字列を入力欄に記載し、「削除」をクリックする

以下を確認すること
- 設定画面が表示されること
- 「このコンポーネントを削除してもよろしいですか？」のダイアログが表示されること
- プロジェクトダッシュボードの「コンポーネント」から該当コンポーネントが消えること

In [ ]:
async def _step(page):
    component_link = page.get_by_role("heading", name=rdm_component_name).get_by_role("link")
    await component_link.click()
    await expect(page.locator('span#nodeTitleEditable')).to_contain_text(rdm_component_name, timeout=transition_timeout)
    if not delete_project:
        return
    await scripts.grdm.delete_project(page, is_component=True)
    await expect(page.locator('span#nodeTitleEditable')).to_contain_text(rdm_project_name, timeout=transition_timeout)

await run_pw(_step)

## プロジェクトダッシュボードの画面右部の「コンポーネント」の「コンポーネントを追加」をクリックする

- 「新しいコンポーネントを作成する」ダイアログが表示されること

In [ ]:
async def _step(page):
    # Click the "コンポーネントを追加" (Add Component) button
    add_component_button = page.locator('div[data-toggle="modal"][data-target="#addSubComponent"]')
    await expect(add_component_button).to_be_visible(timeout=transition_timeout)
    await add_component_button.click()
    
    # Verify the "新しいコンポーネントを作成する" (Create New Component) dialog appears
    await asyncio.sleep(1)
    await expect(page.locator('text=新しいコンポーネントを作成する')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## タイトル「TEST-グループ管理連携機能検証-YYYYMMDD(本日の日付)-コンポーネント-ユーザー2」でコンポーネントを作成する
※上記のコンポーネントを作成する際、「次からメンバーとグループを追加する：{プロジェクト名}」にチェックを入れること

- 「新しいコンポーネントが作成されました！」ダイアログが表示されること

In [ ]:
rdm_component_name = 'TEST-グループ管理連携機能検証-{}-コンポーネント-ユーザー2'.format(datetime.now().strftime('%Y%m%d'))

async def _step(page):
    # Input the component name
    project_name_input = page.locator('#addSubComponent input.form-control.project-name[name="projectName"]')
    await expect(project_name_input).to_be_visible(timeout=transition_timeout)
    await project_name_input.type(rdm_component_name)
    await asyncio.sleep(1)
    await page.locator('input[name="inherit_contributors"][type="checkbox"]').check()
    # Click the save button (作成)
    save_button = page.locator('#addSubComponent button.btn-success:has-text("作成")')
    await expect(save_button).to_be_enabled(timeout=transition_timeout)
    await save_button.click()
    
    # Verify the success dialog appears
    success_dialog = page.locator('text=新しいコンポーネントが作成されました！')
    await expect(success_dialog).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「新しいコンポーネントが作成されました！」ダイアログの「新しいコンポーネントへ移動する」をクリックする

- 作成したコンポーネントのプロジェクトダッシュボードが表示されること

In [ ]:
async def _step(page):
    # Click "新しいコンポーネントへ移動する" button in the success dialog
    go_to_component_link = page.locator('a.btn.btn-success:has-text("新しいコンポーネントへ移動する")')
    await expect(go_to_component_link).to_be_visible(timeout=transition_timeout)
    await go_to_component_link.click()

    # Wait for navigation to the new component's dashboard
    await page.wait_for_load_state('networkidle')
    
    # Verify that the component's project dashboard is displayed
    # Check for the component name in the title
    await expect(page.locator('span#nodeTitleEditable')).to_contain_text(rdm_component_name, timeout=transition_timeout)
await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから「アドオン」をクリックする

- アドオン設定画面が表示されること

In [ ]:
async def _step(page):
    await page.locator("#projectSubnav").get_by_role("link", name="アドオン").click()
    await expect(page.get_by_role("heading", name="アドオンを選択", level=3)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「アドオンを選択」のパネル内「Groups」の行の「有効にする」をクリックする。

・「Groups アドオン規約」のダイアログが表示されること

In [ ]:
async def _step(page):
    enable_button = page.locator('div.addon-container[name="groups"] a', has_text="有効にする")
    await enable_button.scroll_into_view_if_needed()
    await enable_button.click()
    await expect(page.get_by_role("heading", name="Groups アドオン規約", level=3)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「確認」をクリックする

- 「アドオンを構成」のパネル内に「Groups」の行が追加されること
-  プロジェクトダッシュボードの上部メニューに 「グループ」が追加されること

In [ ]:
async def _step(page):
    # Locate and click the "確認" button
    confirm_button = page.locator('button[data-bb-handler="confirm"]', has_text="確認")
    await confirm_button.scroll_into_view_if_needed()
    await confirm_button.click()
    await expect(page.locator('div#groupsScope.scripted')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="グループ", exact=True)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 作成したコンポーネントのプロジェクトダッシュボードの上部メニューから「メンバー」をクリックする

- 「メンバー」画面が表示されること
- 各メンバーが以下の通りとなっていること。
    - 機関管理者1 : 管理者
    - 既存ユーザー1 : 管理者
    - 既存ユーザー2 : 管理者
    - 既存ユーザー3 : 読込み/書込み

In [ ]:
async def _step(page):
    await page.locator("#projectSubnav").get_by_role("link", name="メンバー").click()
    await expect(page.get_by_role("heading", name="メンバー", level=3)).to_be_visible(timeout=transition_timeout)

    # User admin- should still be visible
    member_link = page.get_by_role("link", name=display_username_institutional_admin)
    await expect(member_link).to_be_visible(timeout=transition_timeout)
    parent_row = member_link.locator("xpath=ancestor::tr[1]")
    await expect(parent_row.locator("span[data-bind*='permissionText']").first).to_have_text('管理者', timeout=transition_timeout)

    # User 1 - should still be visible
    member_link = page.get_by_role("link", name=display_username_1)
    await expect(member_link).to_be_visible(timeout=transition_timeout)
    parent_row = member_link.locator("xpath=ancestor::tr[1]")
    await expect(parent_row.locator("span[data-bind*='permissionText']").first).to_have_text('管理者', timeout=transition_timeout)

    # User 2 - should still be visible
    member_link = page.get_by_role("link", name=display_username_2)
    await expect(member_link).to_be_visible(timeout=transition_timeout)
    parent_row = member_link.locator("xpath=ancestor::tr[1]")
    await expect(parent_row.locator("span[data-bind*='permissionText']").first).to_have_text('管理者', timeout=transition_timeout)

    # Group D - should still be visible
    member_link = page.get_by_role("link", name=display_username_3)
    await expect(member_link).to_be_visible(timeout=transition_timeout)
    parent_row = member_link.locator("xpath=ancestor::tr[1]")
    await expect(parent_row.locator("span[data-bind*='permissionText']").first).to_have_text('読込み / 書込み', timeout=transition_timeout)

await run_pw(_step)

## 作成したコンポーネントのプロジェクトダッシュボードの上部メニューから「グループ」をクリックする

- 「グループ」画面が表示されること
- 各グループが以下の通りとなっていること。
    - 既存ユーザー2 が所属するグループ(グループA): 読込み/書込み
    - 既存ユーザー3 が所属するグループ(グループB): 読込み
    - 既存ユーザー4 が所属するグループ(グループC): 管理者
    - 既存ユーザー5 が所属するグループ(グループD): 読込み

In [ ]:
async def _step(page):
    await page.locator("#projectSubnav").get_by_role("link", name="グループ", exact=True).click()
    await expect(page.get_by_role("heading", name=group_note_text)).to_be_visible(timeout=transition_timeout)

    # Group A
    # find the group link (use .nth(0) to avoid calling a Locator)
    group_link = page.get_by_role("link", name=group_a)
    await expect(group_link).to_be_visible(timeout=transition_timeout)
    
    # get the table row containing that link
    parent_row = group_link.locator("xpath=ancestor::tr[1]")
    await expect(parent_row).to_be_visible(timeout=transition_timeout)

    # assert the permission text in the same row - target the span specifically
    await expect(parent_row.locator("span[data-bind*='permissionText']").first).to_have_text('読込み / 書込み', timeout=transition_timeout)

    # Group B
    # find the group link (use .nth(0) to avoid calling a Locator)
    group_link = page.get_by_role("link", name=group_b)
    await expect(group_link).to_be_visible(timeout=transition_timeout)
    
    # get the table row containing that link
    parent_row = group_link.locator("xpath=ancestor::tr[1]")
    await expect(parent_row).to_be_visible(timeout=transition_timeout)

    # assert the permission text in the same row - target the span specifically
    await expect(parent_row.locator("span[data-bind*='permissionText']").first).to_have_text('読込み', timeout=transition_timeout)

    # Group C
    # find the group link (use .nth(0) to avoid calling a Locator)
    group_link = page.get_by_role("link", name=group_c)
    await expect(group_link).to_be_visible(timeout=transition_timeout)
    
    # get the table row containing that link
    parent_row = group_link.locator("xpath=ancestor::tr[1]")
    await expect(parent_row).to_be_visible(timeout=transition_timeout)

    # assert the permission text in the same row - target the span specifically
    await expect(parent_row.locator("span[data-bind*='permissionText']").first).to_have_text('管理者', timeout=transition_timeout)

    # Group D
    # find the group link (use .nth(0) to avoid calling a Locator)
    group_link = page.get_by_role("link", name=group_d)
    await expect(group_link).to_be_visible(timeout=transition_timeout)
    
    # get the table row containing that link
    parent_row = group_link.locator("xpath=ancestor::tr[1]")
    await expect(parent_row).to_be_visible(timeout=transition_timeout)

    # assert the permission text in the same row - target the span specifically
    await expect(parent_row.locator("span[data-bind*='permissionText']").first).to_have_text('読込み', timeout=transition_timeout)
await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから「{プロジェクト名}」をクリックする

- プロジェクトダッシュボードが表示されること
- プロジェクトダッシュボードの画面右部の「コンポーネント」の「{コンポーネント名}」の下に、以下のメンバ/グループが表示されること
  - 機関管理者1
  - 既存ユーザー1 
  - 既存ユーザー2 
  - 既存ユーザー3 
  - 既存ユーザー2 が所属するグループ(グループA)
  - 既存ユーザー3 が所属するグループ(グループB)
  - 既存ユーザー4 が所属するグループ(グループC)
  - 既存ユーザー5 が所属するグループ(グループD)

※ メンバの表示件数は 3 件まで。4 名以上の場合、「あとn人」が表示されること。   
※ グループの表示件数は 3 件まで。4 グループ以上の場合、「あとnグループ」が表示されること。  

In [ ]:
async def _step(page):
    # Click project name in top menu to return to project dashboard
    await page.locator('a[data-toggle="tooltip"][title*="TEST-グループ管理連携機能検証"]').click()
    await page.wait_for_load_state('networkidle')
    await asyncio.sleep(1)

    # Wait for components widget to be visible
    components_section = page.locator("span#components, .render-nodes-list, div:has-text('コンポーネント')")
    await expect(components_section.first).to_be_visible(timeout=transition_timeout)
    
    # Verify component exists by checking for any component item
    component_item = components_section.locator("li.list-group-item-node").first
    await expect(component_item).to_be_visible(timeout=transition_timeout)
    
    # Verify component title contains part of the expected text
    component_title = component_item.locator("h4.list-group-item-heading a").first
    await expect(component_title).to_be_visible(timeout=transition_timeout)
    
    # Verify the title contains expected keywords
    title_text = await component_title.text_content()
    assert "コンポーネント" in title_text, f"Expected 'コンポーネント' in title, but got: {title_text}"

    # Verify users
    user_block = component_item.locator(".project-authors").nth(0)
    user_links = user_block.locator("a.overflow")
    # Visible users (max 3)
    visible_users = await user_links.all_text_contents()
    expected_users = {display_username_institutional_admin, display_username_1, display_username_2}
    for user in visible_users:
        assert user in expected_users
    # Hidden users exist
    await expect(user_block.get_by_text("あと1人")).to_be_visible()

    # Verify groups
    group_block = component_item.locator(".project-authors").nth(1)
    group_links = group_block.locator("a.overflow")
    # Visible groups
    visible_group_texts = await group_links.all_text_contents()
    expected_groups = {group_a, group_b, group_c, group_d}
    for group in visible_group_texts:
        assert group in expected_groups
    # Hidden groups exist
    await expect(group_block.get_by_text("あと1グループ")).to_be_visible()

await run_pw(_step)

## ユーザーメニューから「ログアウト」を選択する

・GRDMトップページが表示されること

In [ ]:
async def _step(page):
    await grdm.logout(page, idp_name_1, transition_timeout=transition_timeout)

await run_pw(_step)

## ウェブブラウザの新規プライベートウィンドウで GRDM トップページを表示する

- GRDM トップページが表示されること

In [ ]:
async def _step(page):
    await page.goto(rdm_url)
    await asyncio.sleep(1)

    # 同意する が表示されなくなったことを確認
    await expect(page.locator('//button[text() = "同意する"]')).to_have_count(0, timeout=500)

await run_pw(_step)

## RDMIdPを利用し、既存ユーザー4としてログインする

- GRDM ダッシュボードが表示されること
- 「プロジェクトに対するグループ機能①」シートのNo.4で追加したプロジェクトが表示されていること
- 該当プロジェクトの「Groups」項目に既存ユーザー4が所属しているグループ(グループC)が表示されていること

※ メンバの表示件数は 3 件まで。4 名以上の場合、「他n名」が表示されること。  
※ グループの表示件数は 3 件まで。4 グループ以上の場合、「他nグループ」が表示されること。  

In [ ]:
async def _step(page):
    await scripts.grdm.login_after_logout(page, idp_name_1, idp_username_4, idp_password_4, transition_timeout=transition_timeout)

    project_locator = page.locator('[data-test-dashboard-item]', has=page.locator('[data-test-dashboard-item-title]', has_text=rdm_project_name)).first
    await expect(project_locator).to_be_visible(timeout=transition_timeout)
    # await expect(project_locator).to_contain_text(group_c, timeout=transition_timeout)
    for attempt in range(5):
        try:
            await expect(project_locator).to_contain_text('他1グループ', timeout=transition_timeout)
            break
        except AssertionError:
            if attempt == 4:
                raise
            await page.reload()
            await asyncio.sleep(2)

await run_pw(_step)

## ダッシュボードから「プロジェクトに対するグループ機能①」シートのNo.4で作成したプロジェクトをクリックする

- プロジェクトダッシュボードが表示されること
- プロジェクトダッシュボードの上部メニューに「ファイル」、「Wiki」、「メタデータ」、「エクスポート」、「メンバ」、「グループ」、「アドオン」、「設定」「証跡管理」が表示されること(「管理者」権限)

In [ ]:
async def _step(page):
    await page.locator(f'//*[@data-test-dashboard-item-title and text()="{rdm_project_name}"]').click()

    await expect(page.locator("#projectSubnav").get_by_role("link", name="ファイル")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="Wiki")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="メタデータ")).to_be_visible(timeout=transition_timeout)
    # await expect(page.locator("#projectSubnav").get_by_role("link", name="エクスポート")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="メンバー")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="グループ", exact=True)).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="アドオン")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="設定")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="証跡管理")).to_be_visible(timeout=transition_timeout)
    await expect(grdm.get_select_expanded_storage_title_locator(page, target_storage_name)).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(1)

await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから「グループ」をクリックする

- 「グループ」画面が表示されること
- 各グループが以下の通りとなっていること。
  - 既存ユーザー2 が所属するグループ(グループA): 読込み/書込み
  - 既存ユーザー3 が所属するグループ(グループB): 読込み
  - 既存ユーザー4 が所属するグループ(グループC): 管理者
  - 既存ユーザー5 が所属するグループ(グループD): 読込み

In [ ]:
async def _step(page):
    await page.locator("#projectSubnav").get_by_role("link", name="グループ", exact=True).click()
    await expect(page.get_by_role("heading", name=group_note_text)).to_be_visible(timeout=transition_timeout)

    # Group A
    # find the group link (use .nth(0) to avoid calling a Locator)
    group_link = page.get_by_role("link", name=group_a)
    await expect(group_link).to_be_visible(timeout=transition_timeout)
    
    # get the table row containing that link
    parent_row = group_link.locator("xpath=ancestor::tr[1]")
    await expect(parent_row).to_be_visible(timeout=transition_timeout)

    # assert the permission text in the same row - target the span specifically
    await expect(parent_row.locator("span[data-bind*='permissionText']").first).to_have_text('読込み / 書込み', timeout=transition_timeout)

    # Group B
    # find the group link (use .nth(0) to avoid calling a Locator)
    group_link = page.get_by_role("link", name=group_b)
    await expect(group_link).to_be_visible(timeout=transition_timeout)
    
    # get the table row containing that link
    parent_row = group_link.locator("xpath=ancestor::tr[1]")
    await expect(parent_row).to_be_visible(timeout=transition_timeout)

    # assert the permission text in the same row - target the span specifically
    await expect(parent_row.locator("span[data-bind*='permissionText']").first).to_have_text('読込み', timeout=transition_timeout)

    # Group C
    # find the group link (use .nth(0) to avoid calling a Locator)
    group_link = page.get_by_role("link", name=group_c)
    await expect(group_link).to_be_visible(timeout=transition_timeout)
    
    # get the table row containing that link
    parent_row = group_link.locator("xpath=ancestor::tr[1]")
    await expect(parent_row).to_be_visible(timeout=transition_timeout)

    # assert the permission text in the same row - target the span specifically
    await expect(parent_row.locator("span[data-bind*='permissionText']").first).to_have_text('管理者', timeout=transition_timeout)

    # Group D
    # find the group link (use .nth(0) to avoid calling a Locator)
    group_link = page.get_by_role("link", name=group_d)
    await expect(group_link).to_be_visible(timeout=transition_timeout)
    
    # get the table row containing that link
    parent_row = group_link.locator("xpath=ancestor::tr[1]")
    await expect(parent_row).to_be_visible(timeout=transition_timeout)

    # assert the permission text in the same row - target the span specifically
    await expect(parent_row.locator("span[data-bind*='permissionText']").first).to_have_text('読込み', timeout=transition_timeout)
await run_pw(_step)

## 「名前」項目の既存ユーザー5が所属するグループ(グループD)の列の右端の「×」ボタンをクリックする

- 「グループを削除」ダイアログが表示されること

In [ ]:
async def _step(page):
    # Click the remove button (×) for Group B
    group_d_link = page.get_by_role("link", name=group_d)
    group_d_row = group_d_link.locator("xpath=ancestor::tr[1]")
    remove_button = group_d_row.locator('span[data-bind*="remove"] i.fa-times')
    await asyncio.sleep(1)
    await remove_button.click()

    # Verify the "グループを削除" dialog appears
    await asyncio.sleep(1)
    await expect(page.locator('text=グループを削除')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「結果」の一覧の既存ユーザー2が所属するグループ(グループA)の横の「＋」ボタンをクリックする

- 「追加中」の一覧に既存ユーザー2が所属するグループ(グループA)が表示されること

In [ ]:
async def _step(page):
    # Verify the "グループを削除" dialog appears again
    await expect(page.locator('#removeGroup')).to_be_visible(timeout=transition_timeout)

    # Click the Remove button in the dialog
    await page.locator('#removeGroup a.btn-danger[data-bind*="submit"]').click()
    await asyncio.sleep(1)
    # Verify the Groups page is displayed
    await expect(page.get_by_role("heading", name=group_note_text)).to_be_visible(timeout=transition_timeout)

    # Verify Group B is removed and other groups remain with correct permissions
    # Group A - should still be visible
    group_link = page.get_by_role("link", name=group_a)
    await expect(group_link).to_be_visible(timeout=transition_timeout)
    parent_row = group_link.locator("xpath=ancestor::tr[1]")
    await expect(parent_row.locator("span[data-bind*='permissionText']").first).to_have_text('読込み / 書込み', timeout=transition_timeout)

    # Group B - should still be visible
    group_link = page.get_by_role("link", name=group_b)
    await expect(group_link).to_be_visible(timeout=transition_timeout)
    parent_row = group_link.locator("xpath=ancestor::tr[1]")
    await expect(parent_row.locator("span[data-bind*='permissionText']").first).to_have_text('読込み', timeout=transition_timeout)

    # Group C - should still be visible
    group_link = page.get_by_role("link", name=group_c)
    await expect(group_link).to_be_visible(timeout=transition_timeout)
    parent_row = group_link.locator("xpath=ancestor::tr[1]")
    await expect(parent_row.locator("span[data-bind*='permissionText']").first).to_have_text('管理者', timeout=transition_timeout)


await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから「{プロジェクト名}」をクリックする

- プロジェクトダッシュボードが表示されること
- プロジェクトダッシュボードの画面右部の「コンポーネント」の「{コンポーネント名}」の下に、以下のメンバ/グループが表示されること
  - 機関管理者1
  - 既存ユーザー1 
  - 既存ユーザー2 
  - 既存ユーザー3 
  - 既存ユーザー2 が所属するグループ(グループA)
  - 既存ユーザー3 が所属するグループ(グループB)
  - 既存ユーザー4 が所属するグループ(グループC)
  - 既存ユーザー5 が所属するグループ(グループD)

※ メンバの表示件数は 3 件まで。4 名以上の場合、「あとn人」が表示されること。  
※ グループの表示件数は 3 件まで。4 グループ以上の場合、「あとnグループ」が表示されること。  


In [ ]:
async def _step(page):
    # Click project name in top menu to return to project dashboard
    await page.locator(f'a.project-title:has-text("{rdm_project_name}")').click()
    await page.wait_for_load_state('networkidle')
    await asyncio.sleep(1)

    # Wait for components widget to be visible
    components_section = page.locator("span#components, .render-nodes-list, div:has-text('コンポーネント')")
    await expect(components_section.first).to_be_visible(timeout=transition_timeout)
    
    # Verify component exists by checking for any component item
    component_item = components_section.locator("li.list-group-item-node").first
    await expect(component_item).to_be_visible(timeout=transition_timeout)
    
    # Verify component title contains part of the expected text
    component_title = component_item.locator("h4.list-group-item-heading a").first
    await expect(component_title).to_be_visible(timeout=transition_timeout)
    
    # Verify the title contains expected keywords
    title_text = await component_title.text_content()
    assert "コンポーネント" in title_text, f"Expected 'コンポーネント' in title, but got: {title_text}"

    # Verify users
    user_block = component_item.locator(".project-authors").nth(0)
    user_links = user_block.locator("a.overflow")
    # Visible users (max 3)
    visible_users = await user_links.all_text_contents()
    expected_users = {display_username_institutional_admin, display_username_1, display_username_2, display_username_3}
    for user in visible_users:
        assert user in expected_users
    # Hidden users exist
    await expect(user_block.get_by_text("あと1人")).to_be_visible()

    # Verify groups
    group_block = component_item.locator(".project-authors").nth(1)
    group_links = group_block.locator("a.overflow")
    # Visible groups
    visible_group_texts = await group_links.all_text_contents()
    expected_groups = {group_a, group_b, group_c, group_d}
    for group in visible_group_texts:
        assert group in expected_groups
    # Hidden groups exist
    await expect(group_block.get_by_text("あと1グループ")).to_be_visible()

await run_pw(_step)

##  プロジェクトダッシュボードの画面右部の「コンポーネント」の「{コンポーネント名}」をクリックする

- コンポーネントのプロジェクトダッシュボードが表示されること

In [ ]:
async def _step(page):
    component_link = page.get_by_role("heading", name=rdm_component_name).get_by_role("link")
    await component_link.click()
    await expect(page.locator('span#nodeTitleEditable')).to_contain_text(rdm_component_name, timeout=transition_timeout)

await run_pw(_step)

## コンポーネントのプロジェクトダッシュボードの上部メニューから「グループ」をクリックする

- 「グループ」画面が表示されること
- 各グループが以下の通りとなっていること。
  - 既存ユーザー2 が所属するグループ(グループA): 読込み/書込み
  - 既存ユーザー3 が所属するグループ(グループB): 読込み
  - 既存ユーザー4 が所属するグループ(グループC): 管理者
  - 既存ユーザー5 が所属するグループ(グループD): 読込み

In [ ]:
async def _step(page):
    await page.locator("#projectSubnav").get_by_role("link", name="グループ", exact=True).click()
    await expect(page.get_by_role("heading", name=group_note_text)).to_be_visible(timeout=transition_timeout)

    # Group A
    # find the group link (use .nth(0) to avoid calling a Locator)
    group_link = page.get_by_role("link", name=group_a)
    await expect(group_link).to_be_visible(timeout=transition_timeout)
    
    # get the table row containing that link
    parent_row = group_link.locator("xpath=ancestor::tr[1]")
    await expect(parent_row).to_be_visible(timeout=transition_timeout)

    # assert the permission text in the same row - target the span specifically
    await expect(parent_row.locator("span[data-bind*='permissionText']").first).to_have_text('読込み / 書込み', timeout=transition_timeout)

    # Group B
    # find the group link (use .nth(0) to avoid calling a Locator)
    group_link = page.get_by_role("link", name=group_b)
    await expect(group_link).to_be_visible(timeout=transition_timeout)
    
    # get the table row containing that link
    parent_row = group_link.locator("xpath=ancestor::tr[1]")
    await expect(parent_row).to_be_visible(timeout=transition_timeout)

    # assert the permission text in the same row - target the span specifically
    await expect(parent_row.locator("span[data-bind*='permissionText']").first).to_have_text('読込み', timeout=transition_timeout)

    # Group C
    # find the group link (use .nth(0) to avoid calling a Locator)
    group_link = page.get_by_role("link", name=group_c)
    await expect(group_link).to_be_visible(timeout=transition_timeout)
    
    # get the table row containing that link
    parent_row = group_link.locator("xpath=ancestor::tr[1]")
    await expect(parent_row).to_be_visible(timeout=transition_timeout)

    # assert the permission text in the same row - target the span specifically
    await expect(parent_row.locator("span[data-bind*='permissionText']").first).to_have_text('管理者', timeout=transition_timeout)

    # Group D
    # find the group link (use .nth(0) to avoid calling a Locator)
    group_link = page.get_by_role("link", name=group_d)
    await expect(group_link).to_be_visible(timeout=transition_timeout)
    
    # get the table row containing that link
    parent_row = group_link.locator("xpath=ancestor::tr[1]")
    await expect(parent_row).to_be_visible(timeout=transition_timeout)

    # assert the permission text in the same row - target the span specifically
    await expect(parent_row.locator("span[data-bind*='permissionText']").first).to_have_text('読込み', timeout=transition_timeout)
await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから「グループ」をクリックする

- 「グループ」画面が表示されること
- 各グループが以下の通りとなっていること。
  - 既存ユーザー2 が所属するグループ(グループA): 読込み/書込み
  - 既存ユーザー3 が所属するグループ(グループB): 読込み
  - 既存ユーザー4 が所属するグループ(グループC): 管理者

In [ ]:
async def _step(page):
    # Click project name in top menu to return to project dashboard
    await page.locator('a[data-toggle="tooltip"][title*="TEST-グループ管理連携機能検証"]').click()
    await page.wait_for_load_state('networkidle')
    await asyncio.sleep(1)

    await page.locator("#projectSubnav").get_by_role("link", name="グループ", exact=True).click()
    await expect(page.get_by_role("heading", name=group_note_text)).to_be_visible(timeout=transition_timeout)

    # Group A - should still be visible
    group_link = page.get_by_role("link", name=group_a)
    await expect(group_link).to_be_visible(timeout=transition_timeout)
    parent_row = group_link.locator("xpath=ancestor::tr[1]")
    await expect(parent_row.locator("span[data-bind*='permissionText']").first).to_have_text('読込み / 書込み', timeout=transition_timeout)

    # Group B - should still be visible
    group_link = page.get_by_role("link", name=group_b)
    await expect(group_link).to_be_visible(timeout=transition_timeout)
    parent_row = group_link.locator("xpath=ancestor::tr[1]")
    await expect(parent_row.locator("span[data-bind*='permissionText']").first).to_have_text('読込み', timeout=transition_timeout)

    # Group C - should still be visible
    group_link = page.get_by_role("link", name=group_c)
    await expect(group_link).to_be_visible(timeout=transition_timeout)
    parent_row = group_link.locator("xpath=ancestor::tr[1]")
    await expect(parent_row.locator("span[data-bind*='permissionText']").first).to_have_text('管理者', timeout=transition_timeout)


await run_pw(_step)

## 「名前」項目の既存ユーザー3が所属するグループ(グループB)の列の右端の「×」ボタンをクリックする

- 「グループを削除」ダイアログが表示されること

In [ ]:
async def _step(page):
    # Click the remove button (×) for Group B
    group_b_link = page.get_by_role("link", name=group_b)
    group_b_row = group_b_link.locator("xpath=ancestor::tr[1]")
    remove_button = group_b_row.locator('span[data-bind*="remove"] i.fa-times')
    await asyncio.sleep(1)
    await remove_button.click()

    # Verify the "グループを削除" dialog appears
    await asyncio.sleep(1)
    await expect(page.locator('text=グループを削除')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「グループを削除」ダイアログの「{既存ユーザ3が所属するグループ(グループB)}を{プロジェクト名}およびその中のすべてのコンポーネントからを削除します。」のチェックボックスをチェックして、「削除」ボタンをクリックする。

- 「グループを削除」ダイアログが表示されること
- {プロジェクト名}、{コンポーネント名}が削除対象として表示されること

In [ ]:
async def _step(page):
    # await page.locator('a[data-bind*="click: back"]').click()
    await page.locator('input[type="radio"][name="radioBoxGroup"][value="true"]').check()

    # Verify the "グループを削除" dialog appears again
    await expect(page.locator('#removeGroup')).to_be_visible(timeout=transition_timeout)

    await page.locator('a[data-bind*="deleteAllNodes"]').click()
    await expect(page.locator('text=グループを削除')).to_be_visible(timeout=transition_timeout)
    project_title = page.get_by_role("heading", name=rdm_project_name, exact=True)
    await expect(project_title).to_be_visible(timeout=transition_timeout)
    component_title = page.get_by_role("heading", name=rdm_component_name, exact=True)
    await expect(component_title).to_be_visible(timeout=transition_timeout)
await run_pw(_step)

## 「グループを削除」ダイアログの「削除」ボタンをクリックする。

- 「名前」項目から既存ユーザ3が所属するグループ(グループB)が削除されること
- 各グループが以下の通りとなっていること。
  - 既存ユーザー2 が所属するグループ(グループA): 読込み/書込み
  - 既存ユーザー4 が所属するグループ(グループC): 管理者

In [ ]:
async def _step(page):
    # Verify the "グループを削除" dialog appears again
    await expect(page.locator('#removeGroup')).to_be_visible(timeout=transition_timeout)

    # Click the Remove button in the dialog
    await page.locator('#removeGroup a.btn-danger[data-bind*="submit"]').click()
    await asyncio.sleep(1)
    # Verify the Groups page is displayed
    await expect(page.get_by_role("heading", name=group_note_text)).to_be_visible(timeout=transition_timeout)

    # Verify Group B is removed and other groups remain with correct permissions
    # Group A - should still be visible
    group_link = page.get_by_role("link", name=group_a)
    await expect(group_link).to_be_visible(timeout=transition_timeout)
    parent_row = group_link.locator("xpath=ancestor::tr[1]")
    await expect(parent_row.locator("span[data-bind*='permissionText']").first).to_have_text('読込み / 書込み', timeout=transition_timeout)

    # Group C - should still be visible
    group_link = page.get_by_role("link", name=group_c)
    await expect(group_link).to_be_visible(timeout=transition_timeout)
    parent_row = group_link.locator("xpath=ancestor::tr[1]")
    await expect(parent_row.locator("span[data-bind*='permissionText']").first).to_have_text('管理者', timeout=transition_timeout)


await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから「{プロジェクト名}」をクリックする

- プロジェクトダッシュボードが表示されること
- プロジェクトダッシュボードの画面右部の「コンポーネント」の「{コンポーネント名}」の下に、以下のメンバ/グループが表示されること
  - 機関管理者1
  - 既存ユーザー1 
  - 既存ユーザー2 
  - 既存ユーザー3 
  - 既存ユーザー2 が所属するグループ(グループA)
  - 既存ユーザー4 が所属するグループ(グループC)
  - 既存ユーザー5 が所属するグループ(グループD)

※ メンバの表示件数は 3 件まで。4 名以上の場合、「あとn人」が表示されること。   
※ グループの表示件数は 3 件まで。4 グループ以上の場合、「あとnグループ」が表示されること。   

In [ ]:
async def _step(page):
    # Click project name in top menu to return to project dashboard
    await page.locator(f'a.project-title:has-text("{rdm_project_name}")').click()
    await page.wait_for_load_state('networkidle')
    await asyncio.sleep(1)

    # Wait for components widget to be visible
    components_section = page.locator("span#components, .render-nodes-list, div:has-text('コンポーネント')")
    await expect(components_section.first).to_be_visible(timeout=transition_timeout)
    
    # Verify component exists by checking for any component item
    component_item = components_section.locator("li.list-group-item-node").first
    await expect(component_item).to_be_visible(timeout=transition_timeout)
    
    # Verify component title contains part of the expected text
    component_title = component_item.locator("h4.list-group-item-heading a").first
    await expect(component_title).to_be_visible(timeout=transition_timeout)
    
    # Verify the title contains expected keywords
    title_text = await component_title.text_content()
    assert "コンポーネント" in title_text, f"Expected 'コンポーネント' in title, but got: {title_text}"
    
    # Verify users
    user_block = component_item.locator(".project-authors").nth(0)
    user_links = user_block.locator("a.overflow")
    # Visible users (max 3)
    visible_users = await user_links.all_text_contents()
    expected_users = {display_username_institutional_admin, display_username_1, display_username_2}
    for user in visible_users:
        assert user in expected_users
    # Hidden users exist
    await expect(user_block.get_by_text("あと1人")).to_be_visible()

    # Verify groups
    group_block = component_item.locator(".project-authors").nth(1)
    group_links = group_block.locator("a.overflow")
    # Visible groups
    visible_group_texts = await group_links.all_text_contents()
    expected_groups = {group_a, group_b, group_c, group_d}
    for group in visible_group_texts:
        assert group in expected_groups

await run_pw(_step)

## プロジェクトダッシュボードの画面右部の「コンポーネント」の「{コンポーネント名}」をクリックする

- コンポーネントのプロジェクトダッシュボードが表示されること

In [ ]:
async def _step(page):
    component_link = page.get_by_role("heading", name=rdm_component_name).get_by_role("link")
    await component_link.click()
    await expect(page.locator('span#nodeTitleEditable')).to_contain_text(rdm_component_name, timeout=transition_timeout)

await run_pw(_step)

## コンポーネントのプロジェクトダッシュボードの上部メニューから「グループ」をクリックする

- 「グループ」画面が表示されること
- 各グループが以下の通りとなっていること。
  - 既存ユーザー2 が所属するグループ(グループA): 読込み/書込み
  - 既存ユーザー4 が所属するグループ(グループC): 管理者
  - 既存ユーザー5 が所属するグループ(グループD): 読込み

In [ ]:
async def _step(page):
    await page.locator("#projectSubnav").get_by_role("link", name="グループ", exact=True).click()
    await expect(page.get_by_role("heading", name=group_note_text)).to_be_visible(timeout=transition_timeout)

    # Group A
    # find the group link (use .nth(0) to avoid calling a Locator)
    group_link = page.get_by_role("link", name=group_a)
    await expect(group_link).to_be_visible(timeout=transition_timeout)
    
    # get the table row containing that link
    parent_row = group_link.locator("xpath=ancestor::tr[1]")
    await expect(parent_row).to_be_visible(timeout=transition_timeout)

    # assert the permission text in the same row - target the span specifically
    await expect(parent_row.locator("span[data-bind*='permissionText']").first).to_have_text('読込み / 書込み', timeout=transition_timeout)

    # Group C
    # find the group link (use .nth(0) to avoid calling a Locator)
    group_link = page.get_by_role("link", name=group_c)
    await expect(group_link).to_be_visible(timeout=transition_timeout)
    
    # get the table row containing that link
    parent_row = group_link.locator("xpath=ancestor::tr[1]")
    await expect(parent_row).to_be_visible(timeout=transition_timeout)

    # assert the permission text in the same row - target the span specifically
    await expect(parent_row.locator("span[data-bind*='permissionText']").first).to_have_text('管理者', timeout=transition_timeout)

    # Group D
    # find the group link (use .nth(0) to avoid calling a Locator)
    group_link = page.get_by_role("link", name=group_d)
    await expect(group_link).to_be_visible(timeout=transition_timeout)
    
    # get the table row containing that link
    parent_row = group_link.locator("xpath=ancestor::tr[1]")
    await expect(parent_row).to_be_visible(timeout=transition_timeout)

    # assert the permission text in the same row - target the span specifically
    await expect(parent_row.locator("span[data-bind*='permissionText']").first).to_have_text('読込み', timeout=transition_timeout)
await run_pw(_step)

## ユーザーメニューから「ログアウト」を選択する

- GRDMトップページが表示されること

In [ ]:
async def _step(page):
    await grdm.logout(page, idp_name_1, transition_timeout=transition_timeout)

await run_pw(_step)

## ウェブブラウザの新規プライベートウィンドウで GRDM トップページを表示する

- GRDM トップページが表示されること

In [ ]:
async def _step(page):
    await page.goto(rdm_url)
    await asyncio.sleep(1)

    # 同意する が表示されなくなったことを確認
    await expect(page.locator('//button[text() = "同意する"]')).to_have_count(0, timeout=500)

await run_pw(_step)

## RDMIdPを利用し、機関管理者1としてログインする
(試験条件を分かりやすくするために、機関管理者アカウントを利用して、ユーザー画面にログインする)

- GRDM ダッシュボードが表示されること

In [ ]:
async def _step(page):
    await scripts.grdm.login_after_logout(page, idp_name_1, idp_username_institutional_admin, idp_password_institutional_admin, transition_timeout=transition_timeout)
    await asyncio.sleep(1)
    await grdm.expect_dashboard(page, transition_timeout=transition_timeout)
    await expect(page.locator(f'//*[@data-test-dashboard-item-title and text()="{rdm_project_name}"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ダッシュボードから「プロジェクトに対するグループ機能①」シートのNo.4で作成したプロジェクトをクリックする

- プロジェクトダッシュボードが表示されること

In [ ]:
async def _step(page):
    # Click project name in top menu to return to project dashboard
    await page.locator(f'//*[@data-test-dashboard-item-title and text()="{rdm_project_name}"]').click()
    await expect(page.locator('//a[text() = "アドオン"]')).to_be_visible(timeout=transition_timeout)
    await expect(grdm.get_select_expanded_storage_title_locator(page, target_storage_name)).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(1)
await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから「グループ」をクリックする

- 「グループ」画面が表示されること

In [ ]:
async def _step(page):
    await page.locator("#projectSubnav").get_by_role("link", name="グループ", exact=True).click()
    await expect(page.get_by_role("heading", name=group_note_text)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「グループ」のタイトルの横にある「＋追加」をクリックする

- 「グループを追加」ダイアログが表示されること

In [ ]:
async def _step(page):
    await page.locator('a[href="#addGroups"]').click()
    await expect(page.locator('#addGroups')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('#addGroups h3.modal-title')).to_have_text("グループを追加", timeout=transition_timeout)

await run_pw(_step)

## 既存ユーザー3が所属するグループ(グループB)を入力して、「検索」ボタンをクリックする

- 「結果」の一覧に既存ユーザー3が所属するグループ(グループB)が表示されること
- 「結果」の一覧に既存ユーザー5が所属するグループ(グループD)が表示されること
- 「結果」の一覧に既存ユーザー2が所属するグループ(グループA)、既存ユーザー4 が所属するグループ(グループC)が「✔️(追加済)」として表示されること

In [ ]:
async def _step(page):
    await page.fill('#addGroups input[data-bind*="value:query"]', group_search)
    await page.click('#addGroups input[type="submit"]')
    
    # Group A - should show as already added with checkmark
    row = page.locator('#addGroups tbody tr', has_text=group_a)
    await expect(row).to_be_visible(timeout=transition_timeout)
    await expect(row.locator('span[data-bind*="group.name"]')).to_have_text(group_a, timeout=transition_timeout)
    # Verify the checkmark icon is visible (already added indicator)
    await expect(row.locator('i.fa-check-circle-o, i.fa-check')).to_be_visible(timeout=transition_timeout)
    
    # Group B
    row = page.locator('#addGroups tbody tr', has_text=group_b)
    await expect(row).to_be_visible(timeout=transition_timeout)
    await expect(row.locator('span[data-bind*="group.name"]')).to_have_text(group_b, timeout=transition_timeout)
    
    # Group C
    row = page.locator('#addGroups tbody tr', has_text=group_c)
    await expect(row).to_be_visible(timeout=transition_timeout)
    await expect(row.locator('span[data-bind*="group.name"]')).to_have_text(group_c, timeout=transition_timeout)
    # Verify the checkmark icon is visible (already added indicator)
    await expect(row.locator('i.fa-check-circle-o, i.fa-check')).to_be_visible(timeout=transition_timeout)
    
    # Group D
    row = page.locator('#addGroups tbody tr', has_text=group_d)
    await expect(row).to_be_visible(timeout=transition_timeout)
    await expect(row.locator('span[data-bind*="group.name"]')).to_have_text(group_d, timeout=transition_timeout)

await run_pw(_step)

## 「結果」の一覧の既存ユーザー3が所属するグループ(グループB)の横の「＋」ボタンをクリックする

- 「追加中」の一覧に既存ユーザー3が所属するグループ(グループB)が表示されること

In [ ]:
async def _step(page):
    row = page.locator('#addGroups tbody tr', has_text=group_b)
    add_btn = row.locator('a.btn-success.contrib-button.btn-mini')
    await expect(add_btn).to_be_visible(timeout=transition_timeout)
    await add_btn.click()
    modal = page.locator('#addGroups .modal-content')
    await expect(modal).to_be_visible(timeout=transition_timeout)

    # Locate the header span labeled "追加中" (or use "Adding" for English)
    header = modal.locator('span.modal-subheader', has_text='追加中')
    await expect(header).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('#addGroups .modal-body .col-md-8 span', has_text=group_b)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「権限」を「読込み」に設定して、「追加」ボタンをクリックする

- 「コンポーネントを選択」ダイアログが表示されること

In [ ]:
async def _step(page):
    # Locate the selection row on the right (col-md-8) that contains the fullname
    sel_row = page.locator('#addGroups .modal-body .col-md-8 tbody tr', has_text=group_b)
    await expect(sel_row).to_be_visible(timeout=transition_timeout)

    # Locate the permission select inside that row
    permission_select = sel_row.locator('select.form-control.input-sm')

    # Choose the "読込み" option by label (locale-safe)
    await permission_select.select_option(label='読込み')

    # Assert the selected option is the expected label
    await expect(permission_select.locator('option:checked')).to_have_text('読込み', timeout=transition_timeout)
    await page.locator('#addGroups .modal-footer a.btn-primary', has_text='次へ').click()
    await expect(page.locator('#addGroups h3.modal-title')).to_have_text("コンポーネントを選択", timeout=transition_timeout)
await run_pw(_step)

## 「コンポーネントを選択」ダイアログに表示された{コンポーネント名}にチェックを入れずに、「追加」ボタンをクリックする

- 「グループ」画面に既存ユーザー3が所属するグループ(グループB)が追加されること
- 「グループ」画面に既存ユーザー3が所属するグループ(グループB)の「権限」が「読込み」に設定されていること

In [ ]:
async def _step(page):
    await page.locator('#addGroups .modal-footer a.btn-success', has_text='追加').click()

    # optional: wait for the modal to close (confirm submit)
    await expect(page.locator('#addGroups')).not_to_be_visible(timeout=transition_timeout)

    # find the group link (use .nth(0) to avoid calling a Locator)
    group_link = page.get_by_role("link", name=group_b)
    await expect(group_link).to_be_visible(timeout=transition_timeout)
    
    # get the table row containing that link
    parent_row = group_link.locator("xpath=ancestor::tr[1]")
    await expect(parent_row).to_be_visible(timeout=transition_timeout)

    # assert the permission text in the same row - target the span specifically
    await expect(parent_row.locator("span[data-bind*='permissionText']").first).to_have_text('読込み', timeout=transition_timeout)

await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから「{プロジェクト名}」をクリックする

- プロジェクトダッシュボードが表示されること
- プロジェクトダッシュボードの画面右部の「コンポーネント」の「{コンポーネント名}」の下に、以下のメンバ/グループが表示されること
  - 機関管理者1
  - 既存ユーザー1 
  - 既存ユーザー2 
  - 既存ユーザー3 
  - 既存ユーザー2 が所属するグループ(グループA)
  - 既存ユーザー4 が所属するグループ(グループC)
  - 既存ユーザー5 が所属するグループ(グループD)

※ メンバの表示件数は 3 件まで。4 名以上の場合、「あとn人」が表示されること。   
※ グループの表示件数は 3 件まで。4 グループ以上の場合、「あとnグループ」が表示されること。   

In [ ]:
async def _step(page):
    # Click project name in top menu to return to project dashboard
    await page.locator(f'a.project-title:has-text("{rdm_project_name}")').click()
    await page.wait_for_load_state('networkidle')
    await asyncio.sleep(1)

    # Wait for components widget to be visible
    components_section = page.locator("span#components, .render-nodes-list, div:has-text('コンポーネント')")
    await expect(components_section.first).to_be_visible(timeout=transition_timeout)
    
    # Verify component exists by checking for any component item
    component_item = components_section.locator("li.list-group-item-node").first
    await expect(component_item).to_be_visible(timeout=transition_timeout)
    
    # Verify component title contains part of the expected text
    component_title = component_item.locator("h4.list-group-item-heading a").first
    await expect(component_title).to_be_visible(timeout=transition_timeout)
    
    # Verify the title contains expected keywords
    title_text = await component_title.text_content()
    assert "コンポーネント" in title_text, f"Expected 'コンポーネント' in title, but got: {title_text}"

    # Verify users
    user_block = component_item.locator(".project-authors").nth(0)
    user_links = user_block.locator("a.overflow")
    # Visible users (max 3)
    visible_users = await user_links.all_text_contents()
    expected_users = {display_username_institutional_admin, display_username_1, display_username_2}
    for user in visible_users:
        assert user in expected_users
    # Hidden users exist
    await expect(user_block.get_by_text("あと1人")).to_be_visible()

    # Verify groups
    group_block = component_item.locator(".project-authors").nth(1)
    group_links = group_block.locator("a.overflow")
    # Visible groups
    visible_group_texts = await group_links.all_text_contents()
    expected_groups = {group_a, group_b, group_c, group_d}
    for group in visible_group_texts:
        assert group in expected_groups

await run_pw(_step)

## プロジェクトダッシュボードの画面右部の「コンポーネント」の「{コンポーネント名}」をクリックする

- コンポーネントのプロジェクトダッシュボードが表示されること

In [ ]:
async def _step(page):
    component_link = page.get_by_role("heading", name=rdm_component_name).get_by_role("link")
    await component_link.click()
    await expect(page.locator('span#nodeTitleEditable')).to_contain_text(rdm_component_name, timeout=transition_timeout)

await run_pw(_step)

## コンポーネントのプロジェクトダッシュボードの上部メニューから「グループ」をクリックする

- 「グループ」画面が表示されること
- 各グループが以下の通りとなっていること。
  - 既存ユーザー2 が所属するグループ(グループA): 読込み/書込み
  - 既存ユーザー4 が所属するグループ(グループC): 管理者
  - 既存ユーザー5 が所属するグループ(グループD): 読込み

In [ ]:
async def _step(page):
    await page.locator("#projectSubnav").get_by_role("link", name="グループ", exact=True).click()
    await expect(page.get_by_role("heading", name=group_note_text)).to_be_visible(timeout=transition_timeout)

    # Group A
    # find the group link (use .nth(0) to avoid calling a Locator)
    group_link = page.get_by_role("link", name=group_a)
    await expect(group_link).to_be_visible(timeout=transition_timeout)
    
    # get the table row containing that link
    parent_row = group_link.locator("xpath=ancestor::tr[1]")
    await expect(parent_row).to_be_visible(timeout=transition_timeout)

    # assert the permission text in the same row - target the span specifically
    await expect(parent_row.locator("span[data-bind*='permissionText']").first).to_have_text('読込み / 書込み', timeout=transition_timeout)

    # Group C
    # find the group link (use .nth(0) to avoid calling a Locator)
    group_link = page.get_by_role("link", name=group_c)
    await expect(group_link).to_be_visible(timeout=transition_timeout)
    
    # get the table row containing that link
    parent_row = group_link.locator("xpath=ancestor::tr[1]")
    await expect(parent_row).to_be_visible(timeout=transition_timeout)

    # assert the permission text in the same row - target the span specifically
    await expect(parent_row.locator("span[data-bind*='permissionText']").first).to_have_text('管理者', timeout=transition_timeout)

    # Group D
    # find the group link (use .nth(0) to avoid calling a Locator)
    group_link = page.get_by_role("link", name=group_d)
    await expect(group_link).to_be_visible(timeout=transition_timeout)
    
    # get the table row containing that link
    parent_row = group_link.locator("xpath=ancestor::tr[1]")
    await expect(parent_row).to_be_visible(timeout=transition_timeout)

    # assert the permission text in the same row - target the span specifically
    await expect(parent_row.locator("span[data-bind*='permissionText']").first).to_have_text('読込み', timeout=transition_timeout)
await run_pw(_step)

##  プロジェクトダッシュボードの上部メニューから「{プロジェクト名}」をクリックする

- プロジェクトダッシュボードが表示されること

In [ ]:
async def _step(page):
    # Click project name in top menu to return to project dashboard
    await page.locator('a[data-toggle="tooltip"][title*="TEST-グループ管理連携機能検証"]').click()
    await page.wait_for_load_state('networkidle')
    await asyncio.sleep(1)

await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから「グループ」をクリックする

- 「グループ」画面が表示されること
- 各グループが以下の通りとなっていること。
  - 既存ユーザー2 が所属するグループ(グループA): 読込み/書込み
  - 既存ユーザー3 が所属するグループ(グループB): 読込み
  - 既存ユーザー4 が所属するグループ(グループC): 管理者

In [ ]:
async def _step(page):
    await page.locator("#projectSubnav").get_by_role("link", name="グループ", exact=True).click()
    await expect(page.get_by_role("heading", name=group_note_text)).to_be_visible(timeout=transition_timeout)

    # Group A
    # find the group link (use .nth(0) to avoid calling a Locator)
    group_link = page.get_by_role("link", name=group_a)
    await expect(group_link).to_be_visible(timeout=transition_timeout)
    
    # get the table row containing that link
    parent_row = group_link.locator("xpath=ancestor::tr[1]")
    await expect(parent_row).to_be_visible(timeout=transition_timeout)

    # assert the permission text in the same row - target the span specifically
    await expect(parent_row.locator("span[data-bind*='permissionText']").first).to_have_text('読込み / 書込み', timeout=transition_timeout)

    # Group B
    # find the group link (use .nth(0) to avoid calling a Locator)
    group_link = page.get_by_role("link", name=group_b)
    await expect(group_link).to_be_visible(timeout=transition_timeout)
    
    # get the table row containing that link
    parent_row = group_link.locator("xpath=ancestor::tr[1]")
    await expect(parent_row).to_be_visible(timeout=transition_timeout)

    # assert the permission text in the same row - target the span specifically
    await expect(parent_row.locator("span[data-bind*='permissionText']").first).to_have_text('読込み', timeout=transition_timeout)

    # Group C
    # find the group link (use .nth(0) to avoid calling a Locator)
    group_link = page.get_by_role("link", name=group_c)
    await expect(group_link).to_be_visible(timeout=transition_timeout)

    # get the table row containing that link
    parent_row = group_link.locator("xpath=ancestor::tr[1]")
    await expect(parent_row).to_be_visible(timeout=transition_timeout)

    # assert the permission text in the same row - target the span specifically
    await expect(parent_row.locator("span[data-bind*='permissionText']").first).to_have_text('管理者', timeout=transition_timeout)
await run_pw(_step)

## 「名前」項目の既存ユーザー3が所属するグループ(グループB)の列の右端の「×」ボタンをクリックする

- 「グループを削除」ダイアログが表示されること

In [ ]:
async def _step(page):
    # Click the remove button (×) for Group B
    group_b_link = page.get_by_role("link", name=group_b)
    group_b_row = group_b_link.locator("xpath=ancestor::tr[1]")
    remove_button = group_b_row.locator('span[data-bind*="remove"] i.fa-times')
    await asyncio.sleep(1)
    await remove_button.click()

    # Verify the "グループを削除" dialog appears
    await asyncio.sleep(1)
    await expect(page.locator('text=グループを削除')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「グループを削除」ダイアログの「削除」ボタンをクリックする

- 「名前」項目から既存ユーザ3が所属するグループ(グループB)が削除されること
- 各グループが以下の通りとなっていること。
  - 既存ユーザー2 が所属するグループ(グループA): 読込み/書込み
  - 既存ユーザー4 が所属するグループ(グループC): 管理者

In [ ]:
async def _step(page):
    # Verify the "グループを削除" dialog appears again
    await expect(page.locator('#removeGroup')).to_be_visible(timeout=transition_timeout)

    # Click the Remove button in the dialog
    await page.locator('#removeGroup a.btn-danger[data-bind*="submit"]').click()
    await asyncio.sleep(1)
    # Verify the Groups page is displayed
    await expect(page.get_by_role("heading", name=group_note_text)).to_be_visible(timeout=transition_timeout)

    # Verify Group B is removed and other groups remain with correct permissions
    # Group A - should still be visible
    group_link = page.get_by_role("link", name=group_a)
    await expect(group_link).to_be_visible(timeout=transition_timeout)
    parent_row = group_link.locator("xpath=ancestor::tr[1]")
    await expect(parent_row.locator("span[data-bind*='permissionText']").first).to_have_text('読込み / 書込み', timeout=transition_timeout)

    # Group C - should still be visible
    group_link = page.get_by_role("link", name=group_c)
    await expect(group_link).to_be_visible(timeout=transition_timeout)
    parent_row = group_link.locator("xpath=ancestor::tr[1]")
    await expect(parent_row.locator("span[data-bind*='permissionText']").first).to_have_text('管理者', timeout=transition_timeout)

await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから「グループ」をクリックする

- 「グループ」画面が表示されること

In [ ]:
async def _step(page):
    await page.locator("#projectSubnav").get_by_role("link", name="グループ", exact=True).click()
    await expect(page.get_by_role("heading", name=group_note_text)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「グループ」のタイトルの横にある「＋追加」をクリックする

- 「グループを追加」ダイアログが表示されること

In [ ]:
async def _step(page):
    await page.locator('a[href="#addGroups"]').click()
    await expect(page.locator('#addGroups')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('#addGroups h3.modal-title')).to_have_text("グループを追加", timeout=transition_timeout)

await run_pw(_step)

## 既存ユーザー3が所属するグループ(グループC)を入力して、「検索」ボタンをクリックする

- 「結果」の一覧に既存ユーザー3が所属するグループ(グループB)が表示されること
- 「結果」の一覧に既存ユーザー2が所属するグループ(グループA)、既存ユーザー4が所属するグループ(グループC) が所属するグループが「✔️(追加済)」として表示されること

In [ ]:
async def _step(page):
    await page.fill('#addGroups input[data-bind*="value:query"]', group_search)
    await page.click('#addGroups input[type="submit"]')
    
    # Group A - should show as already added with checkmark
    row = page.locator('#addGroups tbody tr', has_text=group_a)
    await expect(row).to_be_visible(timeout=transition_timeout)
    await expect(row.locator('span[data-bind*="group.name"]')).to_have_text(group_a, timeout=transition_timeout)
    # Verify the checkmark icon is visible (already added indicator)
    await expect(row.locator('i.fa-check-circle-o, i.fa-check')).to_be_visible(timeout=transition_timeout)
    
    # Group B
    row = page.locator('#addGroups tbody tr', has_text=group_b)
    await expect(row).to_be_visible(timeout=transition_timeout)
    await expect(row.locator('span[data-bind*="group.name"]')).to_have_text(group_b, timeout=transition_timeout)
    
    # Group C
    row = page.locator('#addGroups tbody tr', has_text=group_c)
    await expect(row).to_be_visible(timeout=transition_timeout)
    await expect(row.locator('span[data-bind*="group.name"]')).to_have_text(group_c, timeout=transition_timeout)
    # Verify the checkmark icon is visible (already added indicator)
    await expect(row.locator('i.fa-check-circle-o, i.fa-check')).to_be_visible(timeout=transition_timeout)
    
    # Group D
    row = page.locator('#addGroups tbody tr', has_text=group_d)
    await expect(row).to_be_visible(timeout=transition_timeout)
    await expect(row.locator('span[data-bind*="group.name"]')).to_have_text(group_d, timeout=transition_timeout)

await run_pw(_step)

## 「結果」の一覧の既存ユーザー3が所属するグループ(グループB)の横の「＋」ボタンをクリックする

- 「追加中」の一覧に既存ユーザー3が所属するグループ(グループB)が表示されること

In [ ]:
async def _step(page):
    row = page.locator('#addGroups tbody tr', has_text=group_b)
    add_btn = row.locator('a.btn-success.contrib-button.btn-mini')
    await expect(add_btn).to_be_visible(timeout=transition_timeout)
    await add_btn.click()
    modal = page.locator('#addGroups .modal-content')
    await expect(modal).to_be_visible(timeout=transition_timeout)

    # Locate the header span labeled "追加中" (or use "Adding" for English)
    header = modal.locator('span.modal-subheader', has_text='追加中')
    await expect(header).to_be_visible(timeout=transition_timeout)

    await expect(page.locator('#addGroups .modal-body .col-md-8 span', has_text=group_b)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「権限」を「読込み」に設定して、「追加」ボタンをクリックする

- 「コンポーネントを選択」ダイアログが表示されること

In [ ]:
async def _step(page):
    # Locate the selection row on the right (col-md-8) that contains the fullname
    sel_row = page.locator('#addGroups .modal-body .col-md-8 tbody tr', has_text=group_b)
    await expect(sel_row).to_be_visible(timeout=transition_timeout)

    # Locate the permission select inside that row
    permission_select = sel_row.locator('select.form-control.input-sm')

    # Choose the "読込み" option by label (locale-safe)
    await permission_select.select_option(label='読込み')

    # Assert the selected option is the expected label
    await expect(permission_select.locator('option:checked')).to_have_text('読込み', timeout=transition_timeout)
    await page.locator('#addGroups .modal-footer a.btn-primary', has_text='次へ').click()
    await expect(page.locator('#addGroups h3.modal-title')).to_have_text("コンポーネントを選択", timeout=transition_timeout)
await run_pw(_step)

## 「コンポーネントを選択」ダイアログに表示された{コンポーネント名}にチェックを入れて、「追加」ボタンをクリックする

・「グループ」画面に既存ユーザー3が所属するグループ(グループC)が追加されること
・「グループ」画面に既存ユーザー3が所属するグループ(グループC)の「権限」が「読込み」に設定されていること

In [ ]:
async def _step(page):
    row = page.locator("div.tb-row").filter(has_text=rdm_component_name)
    await expect(row).to_have_count(1)
    # Hover to reveal checkbox
    await row.hover()
    checkbox = row.locator('input[type="checkbox"]')
    await checkbox.check()
    await expect(checkbox).to_be_visible()
    await page.locator('#addGroups .modal-footer a.btn-success', has_text='追加').click()

    # optional: wait for the modal to close (confirm submit)
    await expect(page.locator('#addGroups')).not_to_be_visible(timeout=transition_timeout)

    # find the group link (use .nth(0) to avoid calling a Locator)
    group_link = page.get_by_role("link", name=group_b)
    await expect(group_link).to_be_visible(timeout=transition_timeout)
    
    # get the table row containing that link
    parent_row = group_link.locator("xpath=ancestor::tr[1]")
    await expect(parent_row).to_be_visible(timeout=transition_timeout)

    # assert the permission text in the same row - target the span specifically
    await expect(parent_row.locator("span[data-bind*='permissionText']").first).to_have_text('読込み', timeout=transition_timeout)

await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから「{プロジェクト名}」をクリックする

- プロジェクトダッシュボードが表示されること
- プロジェクトダッシュボードの画面右部の「コンポーネント」の「{コンポーネント名}」の下に、以下のメンバ/グループが表示されること
  - 機関管理者1
  - 既存ユーザー1 
  - 既存ユーザー2 
  - 既存ユーザー3 
  - 既存ユーザー2 が所属するグループ(グループA)
  - 既存ユーザー3 が所属するグループ(グループB)
  - 既存ユーザー4 が所属するグループ(グループC)
  - 既存ユーザー5 が所属するグループ(グループD)

※ メンバの表示件数は 3 件まで。4 名以上の場合、「あとn人」が表示されること。   
※ グループの表示件数は 3 件まで。4 グループ以上の場合、「あとnグループ」が表示されること。   

In [ ]:
async def _step(page):
    # Click project name in top menu to return to project dashboard
    await page.locator(f'a.project-title:has-text("{rdm_project_name}")').click()
    await page.wait_for_load_state('networkidle')
    await asyncio.sleep(1)

    # Wait for components widget to be visible
    components_section = page.locator("span#components, .render-nodes-list, div:has-text('コンポーネント')")
    await expect(components_section.first).to_be_visible(timeout=transition_timeout)
    
    # Verify component exists by checking for any component item
    component_item = components_section.locator("li.list-group-item-node").first
    await expect(component_item).to_be_visible(timeout=transition_timeout)
    
    # Verify component title contains part of the expected text
    component_title = component_item.locator("h4.list-group-item-heading a").first
    await expect(component_title).to_be_visible(timeout=transition_timeout)
    
    # Verify the title contains expected keywords
    title_text = await component_title.text_content()
    assert "コンポーネント" in title_text, f"Expected 'コンポーネント' in title, but got: {title_text}"
    
    # Verify users
    user_block = component_item.locator(".project-authors").nth(0)
    user_links = user_block.locator("a.overflow")
    # Visible users (max 3)
    visible_users = await user_links.all_text_contents()
    expected_users = {display_username_institutional_admin, display_username_1, display_username_2}
    for user in visible_users:
        assert user in expected_users
    # Hidden users exist
    await expect(user_block.get_by_text("あと1人")).to_be_visible()

    # Verify groups
    group_block = component_item.locator(".project-authors").nth(1)
    group_links = group_block.locator("a.overflow")
    # Visible groups
    visible_group_texts = await group_links.all_text_contents()
    expected_groups = {group_a, group_b, group_c, group_d}
    for group in visible_group_texts:
        assert group in expected_groups
    # Hidden groups exist
    await expect(group_block.get_by_text("あと1グループ")).to_be_visible()

await run_pw(_step)

## プロジェクトダッシュボードの画面右部の「コンポーネント」の「{コンポーネント名}」をクリックする

- コンポーネントのプロジェクトダッシュボードが表示されること

In [ ]:
async def _step(page):
    component_link = page.get_by_role("heading", name=rdm_component_name).get_by_role("link")
    await component_link.click()
    await expect(page.locator('span#nodeTitleEditable')).to_contain_text(rdm_component_name, timeout=transition_timeout)

await run_pw(_step)

## コンポーネントのプロジェクトダッシュボードの上部メニューから「グループ」をクリックする

- 「グループ」画面が表示されること
- 各グループが以下の通りとなっていること。
  - 既存ユーザー1 が所属するグループ(グループA): 読込み/書込み
  - 既存ユーザー2 が所属するグループ(グループB): 読込み
  - 既存ユーザー4 が所属するグループ(グループC): 管理者
  - 既存ユーザー5 が所属するグループ(グループD): 読込み

In [ ]:
async def _step(page):
    await page.locator("#projectSubnav").get_by_role("link", name="グループ", exact=True).click()
    await expect(page.get_by_role("heading", name=group_note_text)).to_be_visible(timeout=transition_timeout)

    # Group A
    # find the group link (use .nth(0) to avoid calling a Locator)
    group_link = page.get_by_role("link", name=group_a)
    await expect(group_link).to_be_visible(timeout=transition_timeout)
    
    # get the table row containing that link
    parent_row = group_link.locator("xpath=ancestor::tr[1]")
    await expect(parent_row).to_be_visible(timeout=transition_timeout)

    # assert the permission text in the same row - target the span specifically
    await expect(parent_row.locator("span[data-bind*='permissionText']").first).to_have_text('読込み / 書込み', timeout=transition_timeout)

    # Group B
    # find the group link (use .nth(0) to avoid calling a Locator)
    group_link = page.get_by_role("link", name=group_b)
    await expect(group_link).to_be_visible(timeout=transition_timeout)
    
    # get the table row containing that link
    parent_row = group_link.locator("xpath=ancestor::tr[1]")
    await expect(parent_row).to_be_visible(timeout=transition_timeout)

    # assert the permission text in the same row - target the span specifically
    await expect(parent_row.locator("span[data-bind*='permissionText']").first).to_have_text('読込み', timeout=transition_timeout)

    # Group C
    # find the group link (use .nth(0) to avoid calling a Locator)
    group_link = page.get_by_role("link", name=group_c)
    await expect(group_link).to_be_visible(timeout=transition_timeout)

    # get the table row containing that link
    parent_row = group_link.locator("xpath=ancestor::tr[1]")
    await expect(parent_row).to_be_visible(timeout=transition_timeout)

    # assert the permission text in the same row - target the span specifically
    await expect(parent_row.locator("span[data-bind*='permissionText']").first).to_have_text('管理者', timeout=transition_timeout)
    
    # Group D
    # find the group link (use .nth(0) to avoid calling a Locator)
    group_link = page.get_by_role("link", name=group_d)
    await expect(group_link).to_be_visible(timeout=transition_timeout)

    # get the table row containing that link
    parent_row = group_link.locator("xpath=ancestor::tr[1]")
    await expect(parent_row).to_be_visible(timeout=transition_timeout)

    # assert the permission text in the same row - target the span specifically
    await expect(parent_row.locator("span[data-bind*='permissionText']").first).to_have_text('読込み', timeout=transition_timeout)
await run_pw(_step)

終了処理を実施。

In [ ]:
await finish_pw_context()

In [ ]:
!rm -fr {work_dir}